# Interactive plots with panel

`ts_panel` makes an interactive panel (bokeh) with a plot per channel and a selector of channels. Needs:

```
pip install "scdata[plot]"
```

Do not plot too many points (two years at 1 minute): resample with `frequency`. The recommended maximum is about 100000 points.

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

## A device

In [ ]:
import pandas as pd

start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('7D')).strftime('%Y-%m-%d')

device = sc.Device(params=sc.APIParams(id=18154))
device.options.min_date = start
await device.load()
device.process()

To plot months of data, read them from the backups instead (needs `pip install "scdata[storage]"` and `S3_DATA_BUCKET`, see [21_long_processing.ipynb](21_long_processing.ipynb)):

```python
device.load_from_storage(min_date='2026-01-01', channels=['TEMP', 'HUM', 'NOISE_A'])
```

In [ ]:
# channels: shown first (up to max_plots, 3 by default); the others are in the selector
panel = device.ts_panel(frequency='10Min', channels=['TEMP', 'NOISE_A', 'NO2'], width=1000, height=300)
panel

`panel.servable()` serves it with `panel serve`. The data of the quality checks (`device.qc_data`, see [20_health_checks.ipynb](20_health_checks.ipynb)) can be plotted with `plot_qc_data=True`.

## A test

All the devices of a test, by channel:

In [ ]:
test = sc.Test(name='EXAMPLES_PANEL',
               devices=[sc.Device(params=sc.APIParams(id=device_id)) for device_id in [18154, 17592]],
               force_recreate=True)
for item in test.devices:
    item.options.min_date = start
await test.load()

In [ ]:
test.ts_panel(frequency='10Min', width=1000, height=300)